# Kronos LoRA Fine-Tuned – Evaluation (5 Seeds)

Evaluates the fine-tuned **NeoQuasar/Kronos-base** model on energy-sector test data (2021+).  
Parameters are held identical to the zero-shot baseline for scientific comparability.

| Parameter | Value | Note |
|---|---|---|
| `context_steps` | 80 | **Matches zero-shot baseline** |
| `forecast_steps` | 12 | **Matches zero-shot baseline** |
| `stride_steps` | 12 | **Matches zero-shot baseline** |
| `steps` | 120 | **Matches zero-shot baseline** |
| Seeds | 13, 42, 123, 456, 789 | **Matches zero-shot baseline** |
| Test period | 2021-01-01 → today | **Matches zero-shot baseline** |
| `lora_r` | 4 | Optimal from sensitivity analysis |
| `lora_alpha` | 8 | Optimal from sensitivity analysis |
| `lora_dropout` | 0.20 | Optimal from sensitivity analysis |
| `learning_rate` | 3e-4 | Optimal from sensitivity analysis |

**Workflow:**
1. Setup environment
2. Adapter laden (Upload **oder** Re-Training)
3. Evaluation über alle 5 Seeds
4. Cross-Sectional RankIC aggregieren
5. Vergleich Zero-Shot vs. Fine-Tuned (Paired t-Test)
6. Ergebnisse herunterladen

## 1 · Setup: Clone Repository

In [ ]:
import os
from pathlib import Path

REPO_NAME = "ba"
REPO_URL  = "https://github.com/bp571/ba.git"

if not Path(REPO_NAME).exists():
    !git clone {REPO_URL}

os.chdir(REPO_NAME)
print(f"Working directory: {os.getcwd()}")

!git clone https://github.com/shiyu-coder/Kronos.git 02_finetuning/models/Kronos
print("Kronos cloned")

## 2 · Install Dependencies

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn tqdm einops \
    peft transformers huggingface_hub \
    gluonts python-dotenv yfinance scipy

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3 · Adapter laden

**Option A** – Adapter aus vorherigem Training-Notebook hochladen (`adapter_model.safetensors` + `adapter_config.json`)  
**Option B** – Adapter hier neu trainieren (~15 min auf A100)

Führe **nur eine** der beiden folgenden Zellen aus.

In [ ]:
# ── OPTION B: Re-train adapter (~15 min auf A100) ─────────────────────────────
!python 02_finetuning/training/train_kronos_lora.py

ADAPTER_PATH = "models/kronos-lora-finetuned/final"
print(f"\nAdapter path: {ADAPTER_PATH}")

In [ ]:
# Adapter verifizieren
adapter_path = Path(ADAPTER_PATH)
assert adapter_path.exists(), f"Adapter not found: {adapter_path}"
for f in adapter_path.iterdir():
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")
print("✅ Adapter verified")

## 4 · Evaluation über 5 Seeds

Evaluation des fine-tuned Modells mit identischen Parametern zum Zero-Shot Baseline.

In [ ]:
SEEDS = [13, 42, 123, 456, 789]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Seed: {seed}")
    print(f"{'='*60}")
    !python 02_finetuning/evaluation/main_kronos_finetuned.py \
        --adapter-path {ADAPTER_PATH} \
        --seed {seed} \
        --context 80 \
        --forecast 12 \
        --config config/energy_assets_filtered.yaml

## 5 · Cross-Sectional RankIC aggregieren

Berechnet die RankIC-Statistik über alle Seeds und Assets — exakt dieselbe Methodik wie beim Base-Modell.

In [2]:
!python 01_model_comparison/scripts/evaluate_results.py \
    --results-dir 02_finetuning/results/kronos_finetuned

C:\Users\benep\AppData\Local\Programs\Python\Python312\python.exe: can't open file 'c:\\Users\\benep\\Desktop\\BA\\02_finetuning\\01_model_comparison\\scripts\\evaluate_results.py': [Errno 2] No such file or directory


## 6 · Vergleich mit Zero-Shot Baseline (Paired t-Test)

Statistischer Vergleich zwischen Zero-Shot Kronos und dem LoRA Fine-Tuned Modell.  
Beide Modelle werden unter identischen Bedingungen (`c=80`, `f=12`, `stride=12`, 5 Seeds, Testzeitraum 2021+) verglichen.

In [1]:
BASELINE_RESULTS_DIR = "01_model_comparison/results/kronos"
FINETUNED_RESULTS_DIR = "02_finetuning/results/kronos_finetuned"

# Kurze Zusammenfassung der Baseline zur Gegenprüfung
print("Zero-Shot Baseline Statistiken:")
!python 01_model_comparison/scripts/evaluate_results.py \
    --results-dir {BASELINE_RESULTS_DIR}

Zero-Shot Baseline Statistiken:


C:\Users\benep\AppData\Local\Programs\Python\Python312\python.exe: can't open file 'c:\\Users\\benep\\Desktop\\BA\\02_finetuning\\01_model_comparison\\scripts\\evaluate_results.py': [Errno 2] No such file or directory


In [ ]:
!python 01_model_comparison/scripts/compare_models.py \
    --baseline {BASELINE_RESULTS_DIR} \
    --comparison {FINETUNED_RESULTS_DIR} \
    --baseline-name "Kronos Zero-Shot" \
    --comparison-name "Kronos LoRA Fine-Tuned"

## 7 · Download Results

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

zip_name = "kronos_finetuned_eval_5seeds.zip"

with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    results_dir = Path("02_finetuning/results/kronos_finetuned")
    for f in results_dir.rglob("*.json"):
        zf.write(f, arcname=str(f.relative_to(results_dir)))

size_mb = Path(zip_name).stat().st_size / 1024 / 1024
print(f"Archive: {zip_name}  ({size_mb:.1f} MB)")
files.download(zip_name)
print("Downloaded.")